# 02 — R3D-18: обучение и полный проход видео

Исследование окон, обучение по явному RUN_TRAINING, затем отдельный RUN_FULL_SCAN для выбранного полного видео. Реализация находится в ml/.

## Как интерпретировать результат
Обучение и старые метрики — классификация окон вокруг **известных отметок**, а не автоматический поиск события во всём видео.
Согласованный `within_room_unique_video` split: 30 train / 10 validation; точные копии не пересекаются.
Общий кабинет, люди и неизвестные сеансы могут пересекаться. Независимость камер и сеансов не подтверждена.
Validation используется для выбора checkpoint по loss, а не как независимый финальный тест.
Первый smoke-run предсказывал нарушение для всех validation-окон (accuracy 0.5, recall 1.0).
Это проверка работоспособности, не хорошее качество модели и не законченный детектор.
Исторические метрики и provenance сохранены в `experiments/r3d18_smoke_result.md` и прежних `runs/`.

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src/overfitted_underqualified").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите из корня репозитория или notebooks/")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import importlib.util
import json
import os
import subprocess
from datetime import UTC, datetime

import pandas as pd
import yaml
from IPython.display import display

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(ROOT / "requirements-ml.txt"),
    ])

from overfitted_underqualified.ml.plots import plot_history, show_windows
from overfitted_underqualified.ml.train import (
    build_model,
    prepare_training,
    train_prepared,
)


## 1. Конфигурация

YAML задаёт seed, 16 кадров, pretrained preprocessing и заморозку backbone. Для короткой проверки задайте TRAIN_EPOCHS=1. По умолчанию применяется число эпох из YAML. Каждый запуск пишет в новый каталог; старые runs не перезаписываются. Для CPU явно задайте TRAIN_DEVICE=cpu.
По умолчанию обучение выключено; настройки читаются из .env: для нового эксперимента явно задайте RUN_TRAINING=true в корневом .env. В режиме исследования используются сохранённые результаты; число эпох из YAML применяется только при включённом обучении.


In [ ]:
from dotenv import dotenv_values

# Read afresh on every execution; shell variables take precedence over .env.
# Do not inject settings into os.environ: changing .env should take effect on rerun.
settings = {**dotenv_values(ROOT / ".env"), **os.environ}
run_training_value = str(settings.get("RUN_TRAINING", "false")).strip().lower()
if run_training_value not in {"true", "false", "1", "0", "yes", "no", "on", "off"}:
    raise ValueError("RUN_TRAINING must be true/false")
RUN_TRAINING = run_training_value in {"true", "1", "yes", "on"}
EXISTING_RUN = Path(settings.get("EXISTING_RUN") or "runs/r3d18_smoke")
if not EXISTING_RUN.is_absolute():
    EXISTING_RUN = ROOT / EXISTING_RUN


training_config_path = ROOT / (settings.get("TRAIN_CONFIG") or "configs/r3d18.yaml")
config = yaml.safe_load(training_config_path.read_text(encoding="utf-8"))
print("Конфигурация эксперимента:", training_config_path)
run_name = datetime.now(UTC).strftime("r3d18_notebook_%Y%m%dT%H%M%S_%fZ")
config["output_dir"] = (settings.get("TRAIN_OUTPUT_DIR") or str(ROOT / "runs" / run_name))
train_epochs = settings.get("TRAIN_EPOCHS")
if train_epochs:
    config["epochs"] = int(train_epochs)
if settings.get("TRAIN_DEVICE"):
    config["device"] = settings["TRAIN_DEVICE"]
if settings.get("VIDEO_DIR"):
    config["video_dir"] = settings["VIDEO_DIR"]

print("RUN_TRAINING:", RUN_TRAINING, "epochs:", config["epochs"], "device:", config["device"])


## 2. Данные и split

Проверяем SHA-256 реальных файлов, покрытие исходных ID, метки/отметки events.csv и отсутствие пересечения точных копий. События X-4 и X-47 сохраняются как два окна одного канонического видео. Validation не участвует в обновлении весов.

In [ ]:
inspection_config = dict(config)
if not RUN_TRAINING:
    inspection_config["device"] = "cpu"
prepared = prepare_training(inspection_config)
display(pd.crosstab(
    pd.Series([r.split for r in prepared["records"]], name="split"),
    pd.Series([r.label for r in prepared["records"]], name="label"),
))
print("Окон:", {subset: len(dataset) for subset, dataset in prepared["datasets"].items()})
print("Хеш разметки:", prepared["manifest"]["events_sha256"])
print("Хеш split:", prepared["manifest"]["split_sha256"])

window_table = pd.DataFrame([
    {"split": subset, "clip_id": record.clip_id, "label": record.label,
     "center_sec": center, "source_ids": list(record.aliases), "last_frame_sec": record.last_time}
    for subset, dataset in prepared["datasets"].items() for record, center in dataset.samples
])
display(window_table)
display(pd.crosstab(window_table["split"], window_table["label"]))


## 3. Примеры окон

Показаны четыре из 16 выбранных кадров до crop/нормализации. Проверим итоговый тензор pretrained-преобразования.

In [ ]:
train_data = prepared["datasets"]["train"]
example_indices = [next(i for i, (record, _) in enumerate(train_data.samples) if record.label == label) for label in (0, 1)]
show_windows(train_data, example_indices)
print("После preprocessing:", tuple(train_data[0]["video"].shape), "= C,T,H,W")


## 4. Загрузка R3D-18

Pretrained-веса берутся из кэша, при первом запуске скачиваются. Бинарная голова обучается; при freeze_backbone=true параметры backbone и статистики BatchNorm фиксированы.

In [ ]:
model = None
if RUN_TRAINING:
    model = build_model(prepared["config"]["weights"], prepared["config"]["freeze_backbone"])
    print(model.fc)
    print("Обучаемых параметров:", sum(p.numel() for p in model.parameters() if p.requires_grad))
else:
    print("R3D-18: pretrained backbone → Linear(512, 2). Загрузка весов пропущена.")


## 5. Обучение

Используем те же DataLoader, seed и цикл эпох, что CLI. Validation выполняется в eval mode без градиентов. Повторное выполнение этой ячейки с тем же output_dir отклоняется.

In [ ]:
if RUN_TRAINING:
    output_dir = train_prepared(prepared, model)
else:
    output_dir = EXISTING_RUN
    print("Обучение выключено. Анализируем сохранённый запуск:", output_dir)


## 6. Loss и артефакты

Для одного smoke-epoch график содержит по одной точке: по нему нельзя судить о сходимости.

In [ ]:
history = []
if (output_dir / "history.json").is_file():
    history = json.loads((output_dir / "history.json").read_text(encoding="utf-8"))
    saved_config = yaml.safe_load((output_dir / "config.yaml").read_text(encoding="utf-8"))
    display(saved_config)  # Результаты относятся к этой конфигурации, не к новым настройкам выше.
    plot_history(history)
    display(pd.json_normalize(history))
    display(pd.DataFrame({"artifact": [p.name for p in sorted(output_dir.iterdir())]}))
    print("Checkpoint для 03_evaluation:", output_dir / "best.pt")
else:
    print("Сохранённого запуска нет. Укажите EXISTING_RUN либо явно включите обучение.")


## 7. Learning curves и validation выбранного checkpoint
Сопоставляем loss и accuracy по эпохам. Валидационные метрики берём из сохранённой истории;
реализация метрик остаётся в модуле. Лучший checkpoint выбран по validation loss, а не по последней эпохе.

In [ ]:
import matplotlib.pyplot as plt

from overfitted_underqualified.ml.plots import plot_confusion

if history:
    curves = pd.json_normalize(history).set_index("epoch")
    curves[["train.accuracy", "validation.accuracy"]].plot(marker="o", ylabel="Accuracy", ylim=(0, 1))
    plt.show()
    plt.close("all")
    best_epoch = min(history, key=lambda epoch: epoch["validation"]["loss"])
    display(pd.Series(best_epoch["validation"], name=f"best epoch {best_epoch['epoch']}"))
    plot_confusion(best_epoch["validation"])


## 8. Сравнение сохранённых экспериментов
Добавляйте каталоги вручную. Сравниваем только запуски с одинаковыми входами и окнами.
Иная длительность окна или выборка требует отдельного протокола; не смешиваем такие метрики.
Повторы одного smoke-run проверяют воспроизводимость, а не дают независимые оценки качества.

In [ ]:
RUNS_TO_COMPARE = [ROOT / "runs/r3d18_smoke", ROOT / "runs/r3d18_smoke_repeat", output_dir]
comparison = []
reference_manifest = None
for run_dir in dict.fromkeys(RUNS_TO_COMPARE):
    if not all((run_dir / name).is_file() for name in ("history.json", "data_manifest.json", "config.yaml")):
        print("Пропущен незавершённый/отсутствующий запуск:", run_dir)
        continue
    manifest = json.loads((run_dir / "data_manifest.json").read_text(encoding="utf-8"))
    run_config = yaml.safe_load((run_dir / "config.yaml").read_text(encoding="utf-8"))
    signature = {key: manifest[key] for key in ("events_sha256", "split_sha256", "windows")}
    signature["sampling"] = {key: run_config[key] for key in ("frames", "window_sec", "negative_windows", "weights")}
    signature["preprocessing"] = {"version": run_config.get("preprocessing", "kinetics400_v1"), "roi": run_config.get("roi")}
    signature["validation_negative_windows"] = run_config.get("validation_negative_windows", run_config["negative_windows"])
    if reference_manifest is None:
        reference_manifest = signature
    if signature != reference_manifest:
        print("Другой набор данных/окон, не сравниваем напрямую:", run_dir)
        continue
    run_history = json.loads((run_dir / "history.json").read_text(encoding="utf-8"))
    run_config = yaml.safe_load((run_dir / "config.yaml").read_text(encoding="utf-8"))
    best = min(run_history, key=lambda epoch: epoch["validation"]["loss"])
    comparison.append({"run": str(run_dir.relative_to(ROOT)) if run_dir.is_relative_to(ROOT) else str(run_dir),
                       "epoch": best["epoch"], "seed": run_config["seed"],
                       "learning_rate": run_config["learning_rate"],
                       **{key: best["validation"][key] for key in ("loss", "accuracy", "precision", "recall", "f1")}})
display(pd.DataFrame(comparison))


## 9. Выводы эксперимента
Для первого smoke-run высокая полнота сочеталась с предсказанием violation для каждого окна:
accuracy=0.5 не превосходит тривиальное постоянное предсказание на сбалансированной validation.
Одна эпоха подтверждает выполнение pipeline, но не сходимость.

Для следующего эксперимента запишите гипотезу, одно изменяемое условие, seed и новый output_dir.
После запуска дополните вывод: динамика train/validation, типичные ошибки, подтверждена ли гипотеза.
Не подбирайте порог в этом отчёте. Разбор отдельных ошибок — в 03_evaluation.
Перед коммитом очищайте outputs: таблицы и кадры могут содержать банковские данные.

## 10. Покрытие каталога: что можно исследовать
Бинарная голова не распознаёт тип действия. Сопоставление описательных тегов с каталогом — аудит данных,
а не создание multiclass targets. Точные копии считаются одним видео; разные отметки X-4/X-47 сохраняются.
Для гипотезы о наличных на столе сначала проверьте видимость и условия применимости.


In [ ]:
from overfitted_underqualified.rules.coverage import catalog_coverage

coverage = catalog_coverage(ROOT / "data/annotations/events.csv", ROOT / "data/processed/split_proposal.csv")
coverage_table = pd.DataFrame(coverage["rows"])
display(coverage_table[["event_type", "source_records", "canonical_videos", "train_videos", "validation_videos", "distinct_markers", "assessment"]])
display(coverage_table[["event_type", "applicability", "video_cannot_establish"]])
print("Положительные записи вне поддерживаемых типов:", coverage["unmapped_positive_ids"])
print("Подтверждённых multiclass targets нет; автоматическое распознавание типов не обучалось.")


## 11. Полный проход: конфигурация до просмотра результатов
Отдельный протокол: сканер получает только путь видео. Он не читает разметку и не выбирает окна по отметкам.
Фиксированные настройки — configs/full_video.yaml: 4 секунды, шаг 2, 16 кадров, порог 0.5, merge gap 0.
Меняйте настройки как новую гипотезу с отдельным отчётом, не оптимизируйте их на validation.
Полный проход включается RUN_FULL_SCAN=true. Обучение для него не требуется.


In [ ]:
from overfitted_underqualified.ml.inference import ScanConfig, prepare_scanner, evidence_frames
from overfitted_underqualified.ml.evaluate import resolve_checkpoint
from overfitted_underqualified.ml.plots import show_evidence
from IPython.display import Video

RUN_FULL_SCAN = str(settings.get("RUN_FULL_SCAN", "false")).lower() in {"true", "1", "yes", "on"}
full_config = yaml.safe_load((ROOT / (settings.get("FULL_VIDEO_CONFIG") or "configs/full_video.yaml")).read_text(encoding="utf-8"))
scan_config = ScanConfig(**full_config["scan"])
from overfitted_underqualified.ml.dataset import VIDEO_EXTENSIONS
video_choices = sorted(p for p in Path(prepared["config"]["video_dir"]).rglob("*")
                       if p.suffix.lower() in VIDEO_EXTENSIONS)
if not settings.get("SCAN_VIDEO") and not video_choices:
    raise FileNotFoundError("No videos found; set SCAN_VIDEO to an existing file")
scan_video = Path(settings["SCAN_VIDEO"]) if settings.get("SCAN_VIDEO") else video_choices[0]
if not scan_video.is_absolute():
    scan_video = ROOT / scan_video
display(full_config)
print("RUN_FULL_SCAN:", RUN_FULL_SCAN, "Видео:", scan_video)


In [ ]:
scan_result = None
if RUN_FULL_SCAN:
    chosen_checkpoint = (output_dir / "best.pt") if RUN_TRAINING else resolve_checkpoint(settings.get("CHECKPOINT_PATH"), root=ROOT)
    scanner = prepare_scanner(chosen_checkpoint, scan_config, settings.get("EVAL_DEVICE") or "cpu")
    display(Video(filename=str(scan_video), embed=False))  # локальное воспроизведение, без встроенного видео
    scan_result = scanner.scan(scan_video)
    print("Checkpoint SHA-256:", scanner.provenance["checkpoint_sha256"])
    print("Полный проход, секунд:", scan_result["processing_sec"])
    display(pd.DataFrame(scan_result["windows"]))
else:
    print("Полный проход выключен. Задайте RUN_FULL_SCAN=true для выбранного SCAN_VIDEO.")


## 12. Вероятность во времени и кандидаты
Поддержки пересекающихся положительных окон объединяются; один длинный сигнал даёт один кандидат.
Предполагаемое время — центр окна с максимальной вероятностью, при равенстве более раннего.
Это не точное начало действия. `event_type=None`, уверенность в типе и нормативные ссылки отсутствуют.


In [ ]:
if scan_result is not None:
    trace = pd.DataFrame(scan_result["windows"])
    ax = trace.plot(x="center_sec", y="probability", marker=".", figsize=(12, 3), ylim=(0, 1), legend=False)
    ax.axhline(scan_config.threshold, color="red", linestyle="--", label="fixed threshold")
    for candidate in scan_result["candidates"]:
        ax.axvspan(candidate["start_sec"], candidate["end_sec"], color="orange", alpha=0.2)
        ax.axvline(candidate["timestamp_sec"], color="orange")
    ax.set(xlabel="Время, с (nominal FPS)", ylabel="P(violation)")
    plt.show()
    plt.close("all")
    display(pd.DataFrame(scan_result["candidates"]))
    SELECTED_CANDIDATE = 0  # меняйте индекс для локального исследования
    if scan_result["candidates"]:
        candidate = scan_result["candidates"][SELECTED_CANDIDATE]
        frames = evidence_frames(scan_video, candidate["timestamp_sec"])
        show_evidence(frames, "Контекст бинарного срабатывания; тип неизвестен")
        display(pd.DataFrame([{k: v for k, v in frame.items() if k != "image_base64"} for frame in frames]))
    else:
        print("Кандидатов при заданном пороге нет; это не доказательство отсутствия нарушения.")


## 13. Вывод полного прохода
Оцените длительность непрерывных тревог и расположение пиков. Просмотр кадров помогает сформулировать
гипотезы об ошибках, но не подтверждает юридическую квалификацию. Нет автоматической типизации действий.
В 03 отдельно измеряются video-level ошибки и совпадения с известными отметками; часть разметки может
быть неполной. Нужны новые независимые данные и проверенные примеры действий для типизированного Analyzer.
Перед коммитом очистите outputs и execution counts — кадры остаются только в локальной сессии.


## 14. Отдельный эксперимент с типами действий
Бинарная модель сохранена. Для типов используется отдельная многометочная голова R3D-18,
замороженный backbone и 16 кадров с KINETICS400_V1 preprocessing. Пропущенный тип не считается отрицательным.
18.10.2026 владелец подтвердил типы и полноту пакетом; подтверждение записано в action_review.
Режим per_type_v1 использует только явные решения action_review в events.csv: есть / нет / неизвестно.
31 окно пригодно для обучения; восемь train-типов, но только один тип имеет validation-положительный пример.
Два запуска по пять эпох дали F1=0 для оцениваемого типа; отчёт reports/action_training_20261018.md.
Спорные соответствия X-15, X-26, X-38 и X-44 исключены из автоматических целей.
Для просмотра: python experiments/review_action_annotations.py. Ссылка на локальную страницу появится в data/local_artifacts/.
X-4/X-47 остаются двумя отметками одного видео. Все метки читаются из events.csv.


In [ ]:
from datetime import datetime, timezone
from overfitted_underqualified.ml.action_train import read_config, run_action_training
from overfitted_underqualified.ml.action_data import prepare_action_data

action_config = read_config(ROOT / settings.get("ACTION_CONFIG", "configs/actions.yaml"))
action_config["device"] = settings.get("ACTION_DEVICE", "cpu")
action_config["output_dir"] = str(ROOT / "runs" / ("r3d18_actions_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")))
action_records, action_samples, action_audit = prepare_action_data(action_config)
coverage = pd.json_normalize(action_audit["coverage"])
display(coverage)
display(pd.DataFrame(action_audit["unmapped"]))
print("Проверенных окон:", action_audit["reviewed_windows"], "Готовность к обучению:", action_audit["training_ready"])
print("Классы без подтверждённых отрицательных примеров:", action_audit["training_blockers"])
coverage.set_index("event_type")[["train.candidate_positive_points", "validation.candidate_positive_points",
    "train.positive_points", "validation.positive_points"]].plot.barh(
    title="Кандидаты и проверенные точки по типам; это покрытие, не качество", figsize=(10, 6))
plt.show()


### Запуск только после проверки разметки
`RUN_ACTION_TRAINING=true` — отдельный opt-in. Не меняйте флаг полноты разметки ради обхода проверки:
режим per_type_v1 позволяет проверить только нужные типы у конкретных окон. Остальные остаются неизвестными.
Отсутствие во всём видео указывается отдельно после полного просмотра оригинала.
Конфигурация фиксирует SHA-256 CSV и подтверждение проверки. Смена CSV требует новой проверки.
Пять эпох и пороги фиксируются до validation; сохраняется последняя эпоха, а не лучший результат подбора.


In [ ]:
RUN_ACTION_TRAINING = str(settings.get("RUN_ACTION_TRAINING", "false")).lower() == "true"
action_checkpoint = None
if RUN_ACTION_TRAINING:
    action_checkpoint = run_action_training(action_config)
    print("Action checkpoint:", action_checkpoint)
    action_history = json.loads((action_checkpoint.parent / "history.json").read_text(encoding="utf-8"))
    action_curves = pd.json_normalize(action_history).set_index("epoch")
    action_curves[["train.loss", "validation.loss"]].plot(title="Multilabel loss на известных метках")
    plt.show()
    display(pd.DataFrame(action_history[-1]["validation"]["per_class"]))
else:
    print("Обучение типов выключено. Реального action-checkpoint пока нет; P0 остаётся открытой.")


### Вывод по типам
Положительные примеры в обеих выборках есть только для двух типов; у большинства типов 1–2 видео,
а `cash_counted_without_client_present` отсутствует в train. Отрицательные метки по каталогу не подтверждены.
До их проверки нельзя измерить достоверные precision/F1 или подключить recognizer в реальный сервис.
Синтетические тесты проверяют код и контракт, но не распознавание банковских действий.


## 15. Контролируемые улучшения бинарного baseline
`TRAIN_CONFIG` позволяет выбрать сохранённую конфигурацию отрицательных окон,
geometry или ROI. Преобразование берётся из версии, сохранённой в checkpoint;
замена crop у старой модели без переобучения не считается улучшением.
`IMPROVEMENT_STUDY` указывает каталог исследования со сравнением полных видео.
В нём остаются и удачные, и неудачные результаты. Порог — 0.5, допуск времени — 5 s.
Эти validation-ролики уже использовались при разборе ошибок, поэтому оценка исследовательская.


In [ ]:
study_value = settings.get("IMPROVEMENT_STUDY")
if study_value:
    study_dir = ROOT / study_value
    comparison_path = study_dir / "comparison.json"
    if not comparison_path.is_file():
        raise FileNotFoundError(comparison_path)
    study_rows = json.loads(comparison_path.read_text(encoding="utf-8"))
    study_table = pd.DataFrame([{
        "experiment": row["name"], "epoch": row["checkpoint_epoch"],
        "window_F1": row["window_metrics"]["f1"],
        "full_video_F1": row["full_metrics"]["video"]["f1"],
        "full_video_precision": row["full_metrics"]["video"]["precision"],
        "full_video_recall": row["full_metrics"]["video"]["recall"],
        "false_alarm_candidates": row["full_metrics"]["false_alarms"]["candidates"],
        "known_marker_recall": row["full_metrics"]["temporal"]["known_marker_recall"],
    } for row in study_rows]).set_index("experiment")
    display(study_table)
    study_table[["full_video_F1", "full_video_recall", "known_marker_recall"]].plot.bar(
        ylim=(0, 1.05), figsize=(10, 4), title="10 общих validation-видео; порог 0.5; не независимый test")
    plt.ylabel("Доля")
    plt.tight_layout()
    plt.show()
    study_table["false_alarm_candidates"].plot.bar(
        figsize=(10, 3), title="Лишние кандидаты на 5 отрицательных validation-видео")
    plt.ylabel("Число кандидатов")
    plt.tight_layout()
    plt.show()
else:
    print("Для просмотра проведённого исследования задайте IMPROVEMENT_STUDY в .env.")


### Как выбирать следующий кандидат
Window-F1 не заменяет full-video F1. Сопоставляйте пропуски, ложные тревоги и
таймкоды; для 10 видео одна ошибка меняет accuracy на 10 процентных пунктов.
Не объявляйте выигрыш доказанным переносом на новые кабинеты или сеансы.
`window_peaks` может сохранить несколько временных кандидатов, одновременно
увеличив число ложных тревог: сравнивайте обе стороны, а не только marker recall.
Обучение типов действий по-прежнему требует отдельной проверки каталожных меток.


## X-18 / X-2: один тип и проверка геометрии
В X-2 предмет на столе — банкнота по уточнению владельца. Метки и split не менялись.
Портретное видео 464×848 искажалось resize 171×128 и crop 112×112.
Сравнили один тип с прежней геометрией и ROI кассира/клиента, выбранным по train.
Конфигурации: `configs/actions_identity_control.yaml` и `configs/actions_identity_roi.yaml`.
Обе: 30 фиксированных эпох, train-only балансировка, порог 0.7. Frozen-признаки
вычисляются один раз на запуск; validation не участвует в градиентах.
Полный отчёт: `reports/identity_geometry_20261018.md`. Ниже только чтение результатов.


In [ ]:
import json
identity_path = ROOT / "reports/identity_geometry_20261018.json"
identity_study = json.loads(identity_path.read_text(encoding="utf-8")) if identity_path.exists() else None
if identity_study:
    comparison = pd.DataFrame([{
        "variant": r["name"], "train_F1": r["train"]["f1"],
        "validation_F1": r["validation"]["f1"], "X2_score": r["x2_score"],
        "FP_windows": r["validation"]["confusion_matrix"][0][1],
        "FN_windows": r["validation"]["confusion_matrix"][1][0],
        "matched_events": r["temporal"]["matched"], "missed_events": r["temporal"]["missed"]
    } for r in identity_study["results"]]).set_index("variant")
    display(comparison)
else:
    print("Отчёт сравнения пока отсутствует; обучение автоматически не запускается.")


In [ ]:
if identity_study:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for result in identity_study["results"]:
        history = result["history"]
        epochs = [row["epoch"] for row in history]
        axes[0].plot(epochs, [row["train"]["loss"] for row in history], label=result["name"])
        axes[1].plot(epochs, [row["validation"]["loss"] for row in history], label=result["name"])
    for ax, title in zip(axes, ("Train: weighted BCE", "Validation: unweighted BCE")):
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.2)
    plt.tight_layout(); plt.show()


Оба варианта выучили train (F1=1), но пропустили X-2: validation F1=0.
ROI исправляет геометрию, однако прирост распознавания не подтверждён.
Нужны дополнительные независимые положительные видео и проверяемые отрицательные
примеры идентификации. Из одной точки нельзя получить независимые данные
нарезкой соседних кадров. Validation уже использовалась для исследования;
кабинет, люди и неизвестные сеансы могут пересекаться. Основной сервис не переключён.


## Проверка конкретного типа: unsecured_accepted_cash
После визуального разбора X-38 добавлена одна положительная action-метка в validation.
Для X-2/X-31/X-39 проверено отсутствие этого типа в окне. Неясные X-10/X-14/X-16/X-34
маскированы для этого типа, а не считаются отрицательными. Бинарные метки, время и split сохранены.
Конфигурация: `configs/actions_unsecured_cash.yaml`, фиксированные 30 эпох и порог 0.7.
Новые метки получены по кадровым последовательностям в этой сессии; независимой экспертизы нет.


In [ ]:
import json
cash_path = ROOT / "reports/unsecured_cash_20261018.json"
if cash_path.exists():
    cash_result = json.loads(cash_path.read_text(encoding="utf-8"))
    display(pd.DataFrame(cash_result["known_windows"]).set_index("clip_id"))
    display(pd.Series(cash_result["window_metrics"], name="Known validation windows"))
    display(pd.DataFrame([cash_result["temporal"]]))
    display(pd.DataFrame(cash_result["posthoc_error_review"]))
    print("Положительный и отрицательный пример правильно распознаны:",
          cash_result["useful_window_recognition_demonstrated"])
    print("Полный отчёт:", cash_result["evaluation"], "checkpoint:", cash_result["checkpoint_sha256"])
else:
    print("Отчёт ещё не создан; обучение автоматически не запускается.")


Измерение на одном положительном и трёх отрицательных окнах: [[TN,FP],[FN,TP]]=[[3,0],[1,0]], F1=0.
X-38 пропущен; X-2 даёт ложную тревогу около 4 с при полном проходе.
Score X-2 выше X-38: одним снижением порога правильного разделения не получить.
Это проверка с отрицательным результатом, не положительная приёмка P0.
Полная временная разметка отсутствует: window-метрики не заменяют full-video precision/F1.
Нужны дополнительные независимые положительные видео и проверка условий хранения.


## Видимые признаки → история обслуживания
Исследуем три отдельных признака: клиент у стойки, банкноты на столе, передача банкнот кассиру.
Только явно проверенные значения из `events.csv/primitive_review` участвуют в loss.
Две секунды контекста заканчиваются в точке наблюдения; будущие кадры не используются.
Общий вид и крупный план стола сравниваются отдельными сохранёнными экспериментами.
Split остаётся 30/10. Кандидат временной цепочки требует проверки применимости правила.

In [ ]:
import os, json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import dotenv_values
from overfitted_underqualified.ml.primitive_model import read_config as read_primitive_config
primitive_settings = {**dotenv_values(ROOT / '.env'), **os.environ}
primitive_config = read_primitive_config(ROOT / primitive_settings.get('PRIMITIVE_CONFIG', 'configs/primitives_multiview.yaml'))
primitive_run = Path(primitive_config['output_dir'])
display(pd.Series({key: primitive_config[key] for key in
    ('seed', 'window_sec', 'frames', 'epochs', 'threshold', 'sampling', 'roi', 'scan_step_sec')}))

In [ ]:
from overfitted_underqualified.ml.primitive_data import prepare_primitive_data, PrimitiveDataset, PrimitiveTransform, CLASSES
primitive_records, primitive_samples, primitive_audit = prepare_primitive_data(primitive_config)
primitive_coverage = pd.json_normalize(primitive_audit['coverage'])
display(primitive_coverage)
primitive_points = pd.DataFrame([{'clip_id':s.record.clip_id, 'split':s.record.split,
    'time_sec':s.center_sec, 'positive':list(s.positives), 'negative':list(s.negatives)} for s in primitive_samples])
display(primitive_points.groupby('split').agg(points=('time_sec','size'), videos=('clip_id','nunique')))
print('Готовность данных:', primitive_audit['training_ready'])

### Проверка входа модели
Для локального просмотра установите `SHOW_PRIMITIVE_SAMPLES=true`. Показываются последние кадры
реальных causal-окон с нормализацией, обратной Kinetics. Это приватные кадры: очищайте outputs перед Git.

In [ ]:
if primitive_settings.get('SHOW_PRIMITIVE_SAMPLES', 'false').lower() == 'true':
    examples = [next(s for s in primitive_samples if s.record.split=='train' and name in s.positives) for name in CLASSES]
    dataset = PrimitiveDataset(examples, PrimitiveTransform(primitive_config), primitive_config['window_sec'])
    import torch
    views = len(primitive_config.get('view_rois', [primitive_config['roi']]))
    fig, axes = plt.subplots(len(examples), views, figsize=(5*views, 3*len(examples)), squeeze=False)
    mean = torch.tensor([0.43216, 0.394666, 0.37645])[:,None,None]
    std = torch.tensor([0.22803, 0.22145, 0.216989])[:,None,None]
    for i,sample in enumerate(examples):
        videos = dataset[i]['video']; videos = videos.unsqueeze(0) if videos.ndim==4 else videos
        for v,video in enumerate(videos):
            axes[i,v].imshow((video[:,-1]*std+mean).clamp(0,1).permute(1,2,0))
            axes[i,v].set_title(f'{sample.record.clip_id} @ {sample.center_sec:g}s | {CLASSES[i]}'); axes[i,v].axis('off')
    plt.tight_layout(); plt.show()
else:
    print('Приватные кадры скрыты; SHOW_PRIMITIVE_SAMPLES=true включает просмотр.')

In [ ]:
from overfitted_underqualified.ml.primitive_model import run_primitive_training
if primitive_settings.get('RUN_PRIMITIVE_TRAINING', 'false').lower() == 'true':
    experiment_config = dict(primitive_config)
    experiment_config['device'] = primitive_settings.get('PRIMITIVE_DEVICE', primitive_config['device'])
    primitive_run = ROOT / 'runs' / ('primitives_notebook_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    experiment_config['output_dir'] = str(primitive_run)
    primitive_checkpoint = run_primitive_training(experiment_config)
    print('Новый checkpoint:', primitive_checkpoint)
else:
    print('Обучение выключено. Читаем сохранённый запуск:', primitive_run)

In [ ]:
if (primitive_run / 'history.json').exists():
    primitive_history = json.loads((primitive_run / 'history.json').read_text(encoding='utf-8'))
    fig, ax = plt.subplots(figsize=(8,3))
    for subset in ('train','validation'):
        ax.plot([r['epoch'] for r in primitive_history], [r[subset]['loss'] for r in primitive_history], label=subset)
    ax.set(xlabel='Epoch', ylabel='Masked BCE'); ax.legend(); plt.show()
    display(pd.DataFrame(primitive_history[-1]['validation']['per_class']))
    print('Веса:', primitive_run / 'last.pt')
    display(json.loads((primitive_run / 'checks.json').read_text(encoding='utf-8')))
else:
    print('Сохранённого запуска нет. Для нового запуска нужен явный RUN_PRIMITIVE_TRAINING=true.')

### Вывод эксперимента
Успех отдельного признака не равен качеству всей цепочки. Сначала проверяем ошибки каждого выхода,
затем открываем полный scan в 03. Порог не подбирается по validation. Эти же видео уже многократно
просматривались; результаты исследовательские. Текущий сводный результат — `reports/primitive_study_20261018.json`.

## Детектор банкнот: отдельный визуальный признак
  R3D сохраняет распознавание клиента и передачи наличных. Маленькая U-Net учится находить
  лежащие на столе банкноты по рамкам при ширине crop 512 px. Документы, деньги в руках
  и машинке не являются положительными примерами. Рамки задают приближённые маски.
  Текущий запуск — второй этап: 30 эпох с hard negatives после 60 эпох balanced BCE.
  Начальные веса и их SHA указаны в конфигурации; новый optimizer обучается только на train.
  Конфигурация, обучение и артефакты ниже; реализация находится в `ml/banknote_model.py`.
  Новый запуск начинается только при `RUN_BANKNOTE_TRAINING=true`.


In [ ]:
import json, os
from pathlib import Path
from datetime import datetime, timezone
from dotenv import dotenv_values
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from overfitted_underqualified.ml.banknote_model import read_config as read_banknote_config, prepare_banknote_data, train as train_banknotes
banknote_settings = {**dotenv_values(ROOT / '.env'), **os.environ}
banknote_config = read_banknote_config(ROOT / banknote_settings.get('BANKNOTE_CONFIG','configs/banknotes.yaml'))
banknote_config['device'] = banknote_settings.get('BANKNOTE_DEVICE',banknote_config['device'])
banknote_samples, banknote_manifest = prepare_banknote_data(banknote_config)
display(pd.Series({k:banknote_config.get(k) for k in ('training_protocol','seed','device','epochs','learning_rate','initialize_checkpoint','roi','resize_width','threshold','supplemental_variants')}))
display(pd.DataFrame([{'clip_id':s.record.clip_id,'split':s.record.split,'positive':bool(s.boxes)} for s in banknote_samples]).groupby(['split','clip_id']).agg(frames=('positive','size'), positives=('positive','sum')))
print('Рамки относятся к train. Validation-точки используются только в 03; детектор не выбирает по ним эпоху.')


In [ ]:
# Optional local-only visual audit: show a reviewed box and its context.
if banknote_settings.get('SHOW_BANKNOTE_SAMPLES','false').lower() == 'true':
    import numpy as np
    from matplotlib.patches import Rectangle
    from overfitted_underqualified.ml.dataset import decode_frames
    sample = next(s for s in banknote_samples if s.boxes)
    rgb = decode_frames(sample.record, np.array([sample.time_sec]))[0].permute(1,2,0).numpy()
    fig, ax = plt.subplots(figsize=(6,8)); ax.imshow(rgb)
    for x1,y1,x2,y2 in sample.boxes:
        ax.add_patch(Rectangle((x1*rgb.shape[1],y1*rgb.shape[0]),(x2-x1)*rgb.shape[1],(y2-y1)*rgb.shape[0],fill=False,edgecolor='red'))
    ax.set_title(f'{sample.record.clip_id} @ {sample.time_sec:g}s'); ax.axis('off'); plt.show()


In [ ]:
if banknote_settings.get('RUN_BANKNOTE_TRAINING','false').lower() == 'true':
    banknote_config = dict(banknote_config, output_dir=str(ROOT / 'runs' / ('banknotes_notebook_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))))
    banknote_checkpoint = train_banknotes(banknote_config)
else:
    banknote_checkpoint = ROOT / banknote_settings.get('BANKNOTE_CHECKPOINT_PATH',str(Path(banknote_config['output_dir'])/'last.pt'))
    print('Обучение выключено; сохранённый checkpoint:',banknote_checkpoint)
banknote_history = banknote_checkpoint.parent/'history.json'
if banknote_history.exists():
    curves = pd.DataFrame(json.loads(banknote_history.read_text(encoding='utf-8'))).set_index('epoch')
    curves[['train_loss']].plot(title='Banknote detector — train loss'); plt.show()
    display(pd.Series(json.loads((banknote_checkpoint.parent/'checks.json').read_text(encoding='utf-8'))))


### Вывод эксперимента
  Низкий train loss не доказывает качество на видео. В 03 проверяются известные точки,
  полная последовательность, ложные тревоги и пропуски. Новые файлы — аугментации тех же
  train-источников: они не увеличивают число независимых сцен. Существующие runs не переписываются.


## Дополнительные X-видео: бинарное обучение и устойчивость
  Варианты наследуют метку/время и split оригинала из events.csv. Все четыре варианта
  train используются в новом бинарном эксперименте, validation-варианты — только
  после обучения. Копии одного родителя не считаются независимыми наблюдениями.
  S-сценарии сохраняются отдельно и не получают метки X-роликов автоматически.


In [ ]:
import yaml
import pandas as pd
from IPython.display import display
augmented_config_path=ROOT/'configs/r3d18_augmented.yaml'
if augmented_config_path.exists():
    augmented_config=yaml.safe_load(augmented_config_path.read_text(encoding='utf-8'))
    display(pd.Series({k:augmented_config[k] for k in ('epochs','seed','preprocessing','roi','cache_frozen_features','supplemental_manifest')}))
    print('Основной запуск выше использует TRAIN_CONFIG. Обучение только при RUN_TRAINING=true.')


In [ ]:
import json
import pandas as pd
from IPython.display import display,FileLink
augmentation_study_path=ROOT/'reports/binary_augmentation_study_20261018.json'
if augmentation_study_path.exists():
    augmentation_study=json.loads(augmentation_study_path.read_text(encoding='utf-8'))
    comparison=[]
    for model_key in ('previous_metrics','new_metrics'):
        for scope in ('original_video_metrics','transformed_video_metrics'):
            comparison.append({'model':model_key,'scope':scope,**augmentation_study[model_key][scope]})
    display(pd.DataFrame(comparison))
    display(pd.DataFrame(augmentation_study['new_metrics']['per_variant_video_metrics']).T)
    display(FileLink(str(ROOT/(augmentation_study['new_report']+'.zip'))))
    print('Бинарные окна вокруг отметок. Это не метрики локализации или распознавания конкретных типов.')
else:
    print('Сохранённое сравнение аугментаций ещё отсутствует.')


## Контроль числа обновлений и доли аугментаций
 Прежние 20 эпох дали 120 шагов на оригиналах против 560 на полном наборе копий.
 Теперь обе головы начинают одинаково и делают 120 шагов: оригиналы против
 смеси с вероятностью 80% оригиналов и 20% вариантов. Последний checkpoint
 фиксируется до validation; порог 0.5. Это исследование известной development
 validation, не независимый test и не обещание 100% на новых источниках.


In [ ]:
import json,os
from pathlib import Path
from dotenv import dotenv_values
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,FileLink
matched_settings={**dotenv_values(ROOT/'.env'),**os.environ}
matched_report=ROOT/matched_settings.get('MATCHED_STUDY_REPORT','reports/matched_augmentation_study_20261018T025716_174675Z.json')


In [ ]:
# Explicit opt-in; implementation and optimizer live in the Python module.
if matched_settings.get('RUN_MATCHED_TRAINING','false').lower()=='true':
    import yaml
    from overfitted_underqualified.ml.matched_augmentation import run as run_matched_study
    matched_report=run_matched_study(yaml.safe_load((ROOT/'configs/r3d18_matched_augmentation.yaml').read_text(encoding='utf-8')))


In [ ]:
matched_study=json.loads(matched_report.read_text(encoding='utf-8')) if matched_report.exists() else None
if matched_study:
    rows=[]
    for model_name,result in matched_study['results'].items():
        for scope in ('original_video_metrics','transformed_video_metrics'):
            rows.append({'model':model_name,'scope':scope,**result[scope]})
    display(pd.DataFrame(rows))
    print('Оба checkpoint зафиксированы до оценки; accuracy не подгонялась к100%.')
else:
    print('Отчёт контролируемого эксперимента отсутствует:',matched_report)


In [ ]:
if matched_study:
    for model_name,result in matched_study['results'].items():
        history_path=Path(result['checkpoint']).parent/'history.json'
        learning=pd.DataFrame(json.loads(history_path.read_text(encoding='utf-8'))).set_index('step')
        ax=learning.train_batch_loss.plot(alpha=.3,figsize=(9,3),label='batch loss')
        learning.train_batch_loss.rolling(10,min_periods=1).mean().plot(ax=ax,label='mean of10steps')
        ax.set(title=model_name,xlabel='Optimizer step',ylabel='Train CE');ax.legend();plt.show();plt.close('all')
